# Project D — run the saved router + contest upload format

**Default input:** `contest.csv` in this folder (`id`, `article`) — the course contest articles (~9k rows).

1. **Predict** — load the saved router, score those articles → `predicted_leaves` (leaf topic codes per row).
2. **Upload table** — for each leaf, walk **`Root → … → leaf`** in the taxonomy and emit **one row per `(article, id_cat)`** for **every node on those paths** (ancestors + leaves). Shared prefixes across multiple leaves are **deduplicated**. Map codes to **`id_cat`** via `topics.csv`. Add **`pid`**.

**Contest columns:** `pid`, `id`, `id_cat` → load into `upload.ProjectD_contest2026`.

Run this notebook from the **Homework 4** directory (or set `HOMEWORK4` below).

## 1. Configuration

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Optional

import pandas as pd

# Directory containing this homework (topic files, topics.csv, contest.csv)
HOMEWORK4 = Path(".")

# Saved router from training (must contain tree_meta.pkl + edge_models.pkl)
ROUTER_DIR = HOMEWORK4 / "artifacts" / "full_tree_router_full_corpus"

# Contest holdout articles (course file: id, article)
INPUT_CSV = HOMEWORK4 / "contest.csv"
TEXT_COLUMN = "article"

# Raw predict output for this run (id + article + predicted_leaves)
PREDICTIONS_CSV = HOMEWORK4 / "artifacts" / "contest_predictions.csv"

# Long-format upload file
TOPICS_CSV = HOMEWORK4 / "topics.csv"
CONTEST_UPLOAD_CSV = HOMEWORK4 / "artifacts" / "projectd_contest_upload.csv"

# Your MIT ID for column `pid`
PID = 924904750

LEAVES_SEP = ";"

# Set to a small int (e.g. 50) to smoke-test the pipeline; None = all contest rows
MAX_ROWS: Optional[int] = None

# Nodes on the path Root→leaf to omit from the upload (Root is not a real topic label for articles)
SKIP_PATH_NODES = frozenset({"Root"})

## 2. Run inference (load router → predict leaves per row)

In [3]:
import time

from hierarchical_classifier import apply_router_to_dataframe, load_multilabel_router

router = load_multilabel_router(ROUTER_DIR)

df_in = pd.read_csv(INPUT_CSV, dtype={"id": "int64"})
if "id" not in df_in.columns:
    raise ValueError(f"INPUT_CSV must have an 'id' column; got {list(df_in.columns)}")
if TEXT_COLUMN not in df_in.columns:
    raise ValueError(f"Missing text column {TEXT_COLUMN!r}; columns={list(df_in.columns)}")

df_in[TEXT_COLUMN] = df_in[TEXT_COLUMN].fillna("").astype(str)

n_full = len(df_in)
if MAX_ROWS is not None:
    df_in = df_in.head(int(MAX_ROWS)).copy()
    print(f"Using MAX_ROWS={MAX_ROWS} of {n_full} rows from {INPUT_CSV.name}")
else:
    print(f"Loaded {n_full} rows from {INPUT_CSV.resolve()}")

t0 = time.perf_counter()
df_pred = apply_router_to_dataframe(router, df_in, text_column=TEXT_COLUMN, leaves_sep=LEAVES_SEP)
print(f"Inference wall time: {time.perf_counter() - t0:.1f}s")

PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_pred.to_csv(PREDICTIONS_CSV, index=False)
print(f"Wrote {len(df_pred)} rows to {PREDICTIONS_CSV.resolve()}")
print(
    f"Rows with ≥1 leaf: {(df_pred['n_predicted_leaves'] > 0).sum()}  "
    f"| empty: {(df_pred['n_predicted_leaves'] == 0).sum()}"
)
df_pred.head()

Loaded 8921 rows from /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/contest.csv
Inference wall time: 60.9s
Wrote 8921 rows to /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/artifacts/contest_predictions.csv
Rows with ≥1 leaf: 8315  | empty: 606


,id,article,predicted_leaves,n_predicted_leaves
0,602946,budget schaeubl social issu wolfgang block ch...,E211;GPOL,2
1,413808,budget suggest measur measur measur rate rate...,E121,1
2,586022,built industr group group unit stat eu pari t...,C24,1
3,600102,bullet administ minist minist law sep borrow ...,C172;C173,2
4,345681,bullet case case case exchequ americ budget b...,E142,1


## 3. Contest long format (`pid`, `id`, `id_cat`)

Expands each predicted **leaf** to the full **path** in `topics.csv` (all internal nodes + leaf). Uses `PREDICTIONS_CSV` from section 2, or **`full_tree_router_cli.py predict --input contest.csv`**.

In [4]:
from topic_hierarchy import TopicTree, load_topic_tree, read_topics_csv


def load_child_to_id_cat(topics_path: Path) -> dict[str, int]:
    """child topic code -> id_cat (same parse as TopicTree / Project D)."""
    t = read_topics_csv(str(topics_path))
    t["child"] = t["child"].astype(str).str.strip()
    dup = t["child"].duplicated(keep=False)
    if dup.any():
        raise ValueError("Duplicate child codes in topics.csv after strip")
    return {c: int(x) for c, x in zip(t["child"], t["id_cat"])}


def predictions_to_contest_long(
    pred: pd.DataFrame,
    child_to_id_cat: dict[str, int],
    tree: TopicTree,
    pid: int,
    *,
    leaves_col: str = "predicted_leaves",
    leaves_sep: str = ";",
    id_col: str = "id",
    skip_nodes: frozenset = frozenset({"Root"}),
) -> pd.DataFrame:
    """
    For each predicted leaf, add one row per topic on the path Root -> ... -> leaf
    (internal nodes + leaf). Multiple leaves share ancestors; rows are deduplicated by (id, id_cat).
    """
    all_codes = tree.all_nodes()
    rows: list[dict] = []
    unknown_leaf: set[str] = set()
    unknown_node: set[str] = set()

    for _, r in pred.iterrows():
        aid = int(r[id_col])
        raw = r[leaves_col]
        if pd.isna(raw) or str(raw).strip() == "":
            continue
        seen_ic: set[int] = set()
        for leaf in [p.strip() for p in str(raw).split(leaves_sep) if p.strip()]:
            if leaf not in all_codes:
                unknown_leaf.add(leaf)
                continue
            path = tree.path_from_root_to(leaf)
            for node in path:
                if node in skip_nodes:
                    continue
                ic = child_to_id_cat.get(node)
                if ic is None:
                    unknown_node.add(node)
                    continue
                if ic in seen_ic:
                    continue
                seen_ic.add(ic)
                rows.append({"pid": int(pid), "id": aid, "id_cat": int(ic)})

    out = pd.DataFrame(rows)
    if unknown_leaf:
        print(f"WARNING: leaf not in taxonomy (skipped): {sorted(unknown_leaf)[:20]}...")
    if unknown_node:
        print(f"WARNING: path node has no id_cat (skipped): {sorted(unknown_node)[:20]}...")
    if out.empty:
        return out
    return out.drop_duplicates().sort_values(["id", "id_cat"]).reset_index(drop=True)[
        ["pid", "id", "id_cat"]
    ]

In [5]:
child_to_id_cat = load_child_to_id_cat(TOPICS_CSV)
tree = load_topic_tree(str(TOPICS_CSV), traversal_root="Root")
pred_tbl = pd.read_csv(PREDICTIONS_CSV, dtype={"id": "int64"})
contest = predictions_to_contest_long(
    pred_tbl,
    child_to_id_cat,
    tree,
    PID,
    leaves_col="predicted_leaves",
    leaves_sep=LEAVES_SEP,
    skip_nodes=SKIP_PATH_NODES,
)
CONTEST_UPLOAD_CSV.parent.mkdir(parents=True, exist_ok=True)
contest.to_csv(CONTEST_UPLOAD_CSV, index=False)
print(f"Wrote {len(contest)} upload rows to {CONTEST_UPLOAD_CSV.resolve()}")
print(f"Unique article ids in upload: {contest['id'].nunique()} (predictions had {len(pred_tbl)} rows)")
contest.head(15)

Wrote 45186 upload rows to /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/artifacts/projectd_contest_upload.csv
Unique article ids in upload: 8315 (predictions had 8921 rows)


,pid,id,id_cat
0,924904750,219779,84
1,924904750,219779,93
2,924904750,219884,40
3,924904750,219884,44
4,924904750,219884,45
5,924904750,219884,47
6,924904750,219884,48
7,924904750,219884,72
8,924904750,219884,84
9,924904750,219884,93


## 4. Human-readable metrics on `CONTEST_UPLOAD_CSV`

Sanity-check: which **`id_cat`** values appear most often (with topic **code** + **description** from `topics.csv`), label counts per article, and any orphan codes.

In [6]:
from topic_hierarchy import read_topics_csv

# Works standalone: reads the saved upload file (re-run after section 3)
upload = pd.read_csv(
    CONTEST_UPLOAD_CSV,
    dtype={"pid": "int64", "id": "int64", "id_cat": "int64"},
)

topics_tbl = read_topics_csv(str(TOPICS_CSV))
topics_tbl["id_cat"] = topics_tbl["id_cat"].astype(int)
topics_tbl["child"] = topics_tbl["child"].astype(str).str.strip()
topics_tbl["description"] = (
    topics_tbl["description"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
)

n_articles = upload["id"].nunique()
n_rows = len(upload)
n_distinct_cats = upload["id_cat"].nunique()
print(
    f"Upload rows: {n_rows:,}  |  distinct articles: {n_articles:,}  |  distinct id_cat in file: {n_distinct_cats}"
)
print()

per_article = upload.groupby("id", sort=False).size()
print("Labels per article (path-expanded rows):")
print(per_article.describe().to_string())
print()

vc = upload["id_cat"].value_counts()
top = (
    vc.rename_axis("id_cat")
    .reset_index(name="n_assignments")
    .merge(topics_tbl[["id_cat", "child", "description"]], on="id_cat", how="left")
)
orphan = int(top["child"].isna().sum())
if orphan:
    print(f"WARNING: {orphan} id_cat values in upload have no row in topics.csv\n")

print("Top 40 categories by row count (most often on an article):")
display_cols = ["id_cat", "child", "n_assignments", "description"]
print(top[display_cols].head(40).to_string(index=False))
print()

print("Share of all upload rows (top 15 categories):")
top15 = top.head(15).copy()
top15["pct_of_rows"] = (100 * top15["n_assignments"] / n_rows).round(2)
print(top15[["id_cat", "child", "n_assignments", "pct_of_rows"]].to_string(index=False))

Upload rows: 45,186  |  distinct articles: 8,315  |  distinct id_cat in file: 112

Labels per article (path-expanded rows):
count    8315.000000
mean        5.434275
std         2.375979
min         2.000000
25%         4.000000
50%         4.000000
75%         6.000000
max        19.000000

Top 40 categories by row count (most often on an article):
 id_cat child  n_assignments          description
     39  CCAT           3806 CORPORATE/INDUSTRIAL
     72  ECAT           3022            ECONOMICS
      2    C1           2831       No Description
     84  GCAT           2076    GOVERNMENT/SOCIAL
    107    M1           1704       No Description
    117  MCAT           1704              MARKETS
     40    E1           1632       No Description
     73    G1           1214       No Description
     74   G15           1214   EUROPEAN COMMUNITY
    113   M14           1033    COMMODITY MARKETS
     43  E121            876         MONEY SUPPLY
     42   E12            876    MONETARY/ECONOMI

### Notes

- **`contest.csv`** must stay alongside the notebook (or change `INPUT_CSV`). Columns: `id`, `article`.
- **`MAX_ROWS`:** use a small number to test quickly; use `None` for the full contest set.
- **Path expansion:** each predicted leaf is expanded to **all `id_cat`s on `Root → … → leaf`** (after skipping `SKIP_PATH_NODES`, default `{Root}`). Overlapping paths from multiple leaves are **deduped** per article.
- **Empty predictions:** articles with no predicted leaves produce **no** upload rows for that `id`.
- **Environment:** same **Python + scikit-learn** as training when loading the router.
- **CLI:** `python full_tree_router_cli.py predict --router <dir> --input contest.csv --output artifacts/contest_predictions.csv` then run section 3 only.